# Day 62: Time Series Forecasting with ML Features

## An Alternative Approach
Classical models like ARIMA and Prophet model time directly, but another common strategy is to reframe forecasting as an ordinary supervised learning problem: engineer features out of the date and the series' own history (lag values, rolling statistics, calendar features), then hand that table to any regular ML regressor like Random Forest or gradient boosting. This is attractive because it lets you reuse all the familiar ML tooling (feature importance, cross-validation, easy mixing-in of external features like promotions or weather) rather than being limited to what a specialized time series model supports. The important caveat noted here matters and is easy to miss: tree-based models (like Random Forest) predict by averaging values they saw during training, so they cannot extrapolate a trend past the range of values seen in training — if the series has a strong upward trend, a tree-based model will systematically under-predict future growth. That's why this demo intentionally uses a mostly seasonal (oscillating, not trending) series, so a tree-based model is a fair fit for the exercise.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(3)
dates = pd.date_range('2023-01-01', periods=200, freq='D')
# Purely seasonal (weekly-ish cycle, period ~7*2*pi) series with noise, no long-term trend --
# deliberately chosen since tree-based models can't extrapolate a trend (see note above)
values = 50 + 10 * np.sin(np.arange(200) / 7) + np.random.normal(0, 3, 200)
df = pd.DataFrame({'date': dates, 'value': values})

## Creating Lag Features
A "lag feature" is just the value of the series some number of steps in the past, used as an input feature to predict the current value — this is how you smuggle sequential/temporal information into a model that otherwise treats each row independently. Lags of 1, 2, 3, and 7 days let the model see yesterday's, the last few days', and last week's value (useful since our synthetic data has roughly a weekly-scale oscillation). A rolling mean (here, the average of the past 7 days, computed *before* shifting so it doesn't include the current day) smooths out day-to-day noise and gives the model a sense of the recent local level of the series. Because lag/rolling features need prior rows to exist, the first several rows end up with `NaN` values — `dropna()` removes those rows since they don't have complete features.

In [ ]:
# Lag features: value from 1, 2, 3, and 7 days ago, used as predictors for "today"
for lag in [1, 2, 3, 7]:
    df[f'lag_{lag}'] = df['value'].shift(lag)
# Rolling 7-day average of *past* values (shift(1) first so today's value isn't included --
# avoids leaking the target into its own feature)
df['rolling_mean_7'] = df['value'].shift(1).rolling(7).mean()
# Drop rows with NaN lag/rolling values (the first few rows, which don't have enough history)
df = df.dropna().reset_index(drop=True)
print(df.head())

## Adding Date Features
Beyond lags of the series itself, the calendar date carries useful signal on its own — sales might be higher on weekends, traffic might dip in certain months, etc. Extracting these as explicit numeric features (rather than leaving the model to infer them from raw dates, which tree-based models can't use directly anyway) lets the regressor learn these recurring calendar effects. `day_of_week` (0=Monday ... 6=Sunday) captures weekly patterns, and `day_of_year` (1-365/366) captures within-year seasonal patterns — both are cheap to compute and often surprisingly predictive.

In [ ]:
# day_of_week (0=Mon..6=Sun) captures weekly patterns; day_of_year captures within-year seasonality
df['day_of_week'] = df['date'].dt.dayofweek
df['day_of_year'] = df['date'].dt.dayofyear

## Training a Regressor
With lag and calendar features in place, this is now an ordinary tabular regression problem, so any standard ML regressor works — here a `RandomForestRegressor`. The critical difference from typical ML workflows is how we split train/test: a normal random `train_test_split` would let some training rows come from *after* some test rows in time, which means the model could effectively "see the future" via lag features computed from later data — a form of data leakage that makes results look artificially good. Instead we split by time — the first 80% of rows (chronologically earliest) become the training set and the last 20% (most recent) become the test set — so the evaluation honestly reflects the real forecasting scenario of predicting values you haven't seen yet using only what came before.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Use every engineered feature except the raw date and the target itself
feature_cols = [c for c in df.columns if c not in ['date', 'value']]
# Chronological split (not random!) -- train on the earliest 80%, test on the most recent 20%
split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(train[feature_cols], train['value'])
# R^2: fraction of variance in the test values explained by the model (1.0 = perfect, 0 = no better than the mean)
print('Test R2:', model.score(test[feature_cols], test['value']))